In [1]:
#!pip install transformers
#!pip install "transformers[torch]"

In [2]:
import pandas as pd
import numpy as np
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [3]:
train_data = pd.read_csv("samsum-train.csv")
test_data = pd.read_csv("samsum-test.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [4]:
# random sampling
train_data = train_data.sample(n=4000,random_state=42).reset_index(drop=True)
test_data = test_data.sample(n=500,random_state=42).reset_index(drop=True)

# Data Preprocessing

In [5]:
import re
def clean_data(text):
    text= re.sub(r"\r\n"," ",text)   # next_line -> " "
    text= re.sub(r"\s+\n"," ",text)  # space
    text= re.sub(r"<.*?>"," ",text)  # html tag
    text=text.strip().lower()
    return text

In [6]:
train_data["dialogue"]= train_data["dialogue"].apply(clean_data)
train_data["summary"]= train_data["summary"].apply(clean_data)

val_data["dialogue"]= val_data["dialogue"].apply(clean_data)
val_data["summary"]= val_data["summary"].apply(clean_data)

# Tokenization

In [7]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")
# Convert text data into token_id for fine-tuning, because models cannot understand text

In [8]:
def tokenize(data):
    # padding -> add space to reach max_length && truncate -> remove some alphabet to reach max_length
    # tokenizer ->{ "input_ids": [...], "attention_mask": [...]}
    inputs= tokenizer(data["dialogue"], padding="max_length",max_length=512,truncation=True)
    target= tokenizer(data["summary"], padding="max_length",max_length=150,truncation=True) 
    
    inputs["labels"]=target["input_ids"] # Adds summary token IDs as labels
    return inputs

In [9]:
# axis =1 ->Apply the function row-wise on { "dialogue": "...", "summary": "..."}
train_dataset = train_data.apply(tokenize,axis=1).tolist() 
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [10]:
train_dataset[0]  # len(train_dataset[0]["input_ids"]) -> 512

# input id -> dialogue ki token_id
# attention_mask(tell which is actual value and padding value)
# labels -> summmary ki token_id

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

# Working with the Model

In [11]:
# NLP & Generating Text
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [12]:
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [13]:
# Training Arguments
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 6,
    weight_decay=0.01,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    warmup_steps = 500,
)

In [14]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [15]:
#trainer.train()

In [16]:
model.save_pretrained("/saved_summary_model")
tokenizer.save_pretrained("/saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/saved_summary_model\\tokenizer_config.json',
 '/saved_summary_model\\tokenizer.json')

In [17]:
model.from_pretrained("/saved_summary_model")
tokenizer.from_pretrained("/saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5Tokenizer(name_or_path='/saved_summary_model', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, single_word=False, normalized=False,

# Test the core logic for summarization

In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean_data
    
    # tokenizer
    inputs = tokenizer(
        dialogue,
        return_tensors="pt",  # add from chatgpt
        padding="max_length",
        max_length=512,
        truncation=True,
        #return_tensor="pt"
    ).to(device)
    
    # generate the summary => token ids
    model.to(device)
    target = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,  # generate 4 o/p and best one will be answer
        early_stopping = True
    )

    #token_id convert to summary --> decording
    summary = tokenizer.decode(
        target[0],
        skip_special_tokens=True
    )
    return summary
    

In [23]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""
summarize_dialogue(test_dialogue)

'experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact. ensuring fairness and transparency is becoming a key area of research.'